In [1]:
import os
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error
from io import StringIO

DATA_PATH = r'C:\Users\DELL\OneDrive\Desktop\Projects\Data science\netflix\netflix-recommender\data'

def parse_netflix_fast(filepath, sample_every_n=10):
    rows = []
    current_movie = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.endswith(':'):
                current_movie = int(line[:-1])
            else:
                rows.append(line + f',{current_movie}')
    raw = '\n'.join(rows)
    df = pd.read_csv(
        StringIO(raw),
        names=['user_id', 'rating', 'date', 'movie_id'],
        dtype={'user_id': np.int32, 'rating': np.int8, 'movie_id': np.int32}
    )
    df['date'] = pd.to_datetime(df['date'])
    if sample_every_n > 1:
        kept = df['user_id'].unique()[::sample_every_n]
        df = df[df['user_id'].isin(kept)]
    return df

dfs = []
for i in range(1, 5):
    print(f"Loading file {i}...")
    df = parse_netflix_fast(os.path.join(DATA_PATH, f'combined_data_{i}.txt'), sample_every_n=10)
    dfs.append(df)
    print(f"  → {len(df):,} rows")

full_df = pd.concat(dfs, ignore_index=True)

min_ratings = 20
active_users = full_df.groupby('user_id').size()
active_users = active_users[active_users >= min_ratings].index
full_df = full_df[full_df['user_id'].isin(active_users)]
print(f"\nFinal: {len(full_df):,} ratings | {full_df['user_id'].nunique():,} users | {full_df['movie_id'].nunique():,} movies")
print(f"Date range: {full_df['date'].min()} → {full_df['date'].max()}")

Loading file 1...
  → 2,406,128 rows
Loading file 2...
  → 2,694,523 rows
Loading file 3...
  → 2,253,873 rows
Loading file 4...
  → 2,669,534 rows

Final: 9,407,204 ratings | 96,650 users | 17,765 movies
Date range: 1999-12-09 00:00:00 → 2005-12-31 00:00:00


In [2]:
full_df = full_df.sort_values('date')

cutoff = full_df['date'].quantile(0.8)
print(f"Cutoff date: {cutoff}")

train_df = full_df[full_df['date'] <= cutoff].copy()
test_df  = full_df[full_df['date'] >  cutoff].copy()

train_users  = set(train_df['user_id'])
train_movies = set(train_df['movie_id'])

test_df = test_df[
    test_df['user_id'].isin(train_users) &
    test_df['movie_id'].isin(train_movies)
]

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

Cutoff date: 2005-08-03 00:00:00
Train: 7,537,987 | Test: 1,086,209


In [3]:
user_ids  = sorted(train_df['user_id'].unique())
movie_ids = sorted(train_df['movie_id'].unique())

user_index_map  = {uid: i for i, uid in enumerate(user_ids)}
movie_index_map = {mid: i for i, mid in enumerate(movie_ids)}

print(f"Users: {len(user_index_map):,} | Movies: {len(movie_index_map):,}")

Users: 85,337 | Movies: 17,168


In [4]:
def average_precision_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    hits, score = 0, 0.0
    for i, item in enumerate(recommended):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k) if relevant else 0.0

def map_at_k(recommendations_dict, test_data, k=10):
    ap_scores = []
    for user_id, recommended in recommendations_dict.items():
        relevant = set(
            test_data[(test_data['user_id'] == user_id) &
                      (test_data['rating'] >= 3.5)]['movie_id']
        )
        if relevant:
            ap_scores.append(average_precision_at_k(recommended, relevant, k))
    return sum(ap_scores) / len(ap_scores) if ap_scores else 0.0

In [5]:
class SVDModel:
    def __init__(self, n_factors=50, n_epochs=20, lr=0.005, reg=0.02):
        self.n_factors = n_factors
        self.n_epochs  = n_epochs
        self.lr  = lr
        self.reg = reg

    def fit(self, train_df, user_index_map, movie_index_map):
        self.user_index_map  = user_index_map
        self.movie_index_map = movie_index_map

        n_users  = len(user_index_map)
        n_movies = len(movie_index_map)

        self.global_mean   = train_df['rating'].mean()
        self.user_factors  = np.random.normal(0, 0.1, (n_users,  self.n_factors))
        self.movie_factors = np.random.normal(0, 0.1, (n_movies, self.n_factors))
        self.user_bias     = np.zeros(n_users)
        self.movie_bias    = np.zeros(n_movies)

        rows = train_df[['user_id','movie_id','rating']].values
        for epoch in range(self.n_epochs):
            np.random.shuffle(rows)
            for user_id, movie_id, rating in rows:
                u = user_index_map.get(user_id)
                m = movie_index_map.get(movie_id)
                if u is None or m is None:
                    continue
                pred = (self.global_mean + self.user_bias[u] + self.movie_bias[m]
                        + np.dot(self.user_factors[u], self.movie_factors[m]))
                err = rating - pred

                self.user_bias[u]  += self.lr * (err - self.reg * self.user_bias[u])
                self.movie_bias[m] += self.lr * (err - self.reg * self.movie_bias[m])

                uf = self.user_factors[u].copy()
                self.user_factors[u]  += self.lr * (err * self.movie_factors[m] - self.reg * uf)
                self.movie_factors[m] += self.lr * (err * uf - self.reg * self.movie_factors[m])

            print(f"Epoch {epoch+1}/{self.n_epochs} done")

    def predict_rating(self, user_id, movie_id):
        u = self.user_index_map.get(user_id)
        m = self.movie_index_map.get(movie_id)
        if u is None or m is None:
            return self.global_mean
        pred = (self.global_mean + self.user_bias[u] + self.movie_bias[m]
                + np.dot(self.user_factors[u], self.movie_factors[m]))
        return float(np.clip(pred, 1, 5))

svd = SVDModel(n_factors=50, n_epochs=20, lr=0.005, reg=0.02)
svd.fit(train_df, user_index_map, movie_index_map)

Epoch 1/20 done
Epoch 2/20 done
Epoch 3/20 done
Epoch 4/20 done
Epoch 5/20 done
Epoch 6/20 done
Epoch 7/20 done
Epoch 8/20 done
Epoch 9/20 done
Epoch 10/20 done
Epoch 11/20 done
Epoch 12/20 done
Epoch 13/20 done
Epoch 14/20 done
Epoch 15/20 done
Epoch 16/20 done
Epoch 17/20 done
Epoch 18/20 done
Epoch 19/20 done
Epoch 20/20 done


In [6]:
test_sample = test_df.sample(n=1000, random_state=42)

preds, actuals = [], []
for _, row in test_sample.iterrows():
    pred = svd.predict_rating(row['user_id'], row['movie_id'])
    preds.append(pred)
    actuals.append(row['rating'])

svd_rmse = np.sqrt(mean_squared_error(actuals, preds))
print(f"SVD RMSE: {svd_rmse:.4f}")

SVD RMSE: 0.9305


In [7]:
# Pre-compute top 200 popular movies — same as KNN
movie_popularity = train_df.groupby('movie_id').size()
top_200_movies = movie_popularity.nlargest(200).index.tolist()

print("Generating SVD recommendations...")
test_users = test_df['user_id'].unique()[:500]
svd_recs = {}

for i, uid in enumerate(test_users):
    preds = []
    # Only predict unrated movies
    user_rated = set(train_df[train_df['user_id'] == uid]['movie_id'])
    for mid in top_200_movies:
        if mid not in user_rated:
            preds.append((mid, svd.predict_rating(uid, mid)))
    preds.sort(key=lambda x: x[1], reverse=True)
    svd_recs[uid] = [mid for mid, _ in preds[:10]]
    if i % 100 == 0:
        print(f"  {i}/{len(test_users)} done...")

svd_map = map_at_k(svd_recs, test_df)
print(f"SVD MAP@10: {svd_map:.4f}")

Generating SVD recommendations...
  0/500 done...
  100/500 done...
  200/500 done...
  300/500 done...
  400/500 done...
SVD MAP@10: 0.0235


In [8]:
results = pd.DataFrame({
    'Model':  ['User-KNN', 'Item-KNN', 'SVD'],
    'RMSE':   [0.9957,     1.0240,     svd_rmse],
    'MAP@10': [0.0217,     0.0111,     svd_map]
})
print(results.to_string(index=False))

   Model     RMSE   MAP@10
User-KNN 0.995700 0.021700
Item-KNN 1.024000 0.011100
     SVD 0.930517 0.023456
